# AI-Generated Image Detection Using Deep Learning

**Institution**: SREENIDHI UNIVERSITY  
**Department**: Department of Computer Science and Engineering  
**Academic Year**: 2025–2026  
**Team Members**:
- Bojja Sahasra (`2401100CSE0243`)
- Mudigonda Lalitha Sreya (`2401100CSE0264`)
- Rohan Joshua Francis (`2401100CSE0275`)  
**Faculty Guide / Course Coordinator**: Dr. Saritha Bantu, Professor, Dept. of CSE, Sreenidhi University  
**Project Repository**: [GitHub Repository](https://github.com/LalithaSreya/DL-Mini-Project-AI-Image-Detection)  
**Dataset**: [CIFAKE: Real and AI-Generated Synthetic Images](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images)  

---

## 1. Project Overview & Colab Setup Instructions

### What This Notebook Does:
This notebook builds and trains a Convolutional Neural Network detector using **Transfer Learning with MobileNetV2** to distinguish between:
- **`FAKE` (Label `0`)**: AI-generated synthetic images (generated by Stable Diffusion v1.4)
- **`REAL` (Label `1`)**: Authentic natural photographs (from the CIFAR-10 photographic benchmark)

### 🚀 Google Colab GPU Setup (CRITICAL FOR FAST TRAINING):
1. In the top navigation menu of Google Colab, click **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU**.
3. Click **Save**.
4. Run each cell sequentially from top to bottom by clicking the **Play (▶)** button on the left of each code block or pressing `Shift + Enter`.


## 2. Environment Setup & GPU Verification
Install required Python libraries, import standard packages, and check whether an active GPU is detected by TensorFlow.


In [ ]:
# Install dependencies in the Google Colab environment
!pip install -q kagglehub tensorflow numpy matplotlib scikit-learn pillow

import os
import sys
import json
import time
import shutil
from pathlib import Path
from typing import Tuple, List, Dict, Any

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

print("=" * 60)
print("ENVIRONMENT & HARDWARE VERIFICATION")
print("=" * 60)
print("Python Version     :", sys.version.split()[0])
print("TensorFlow Version :", tf.__version__)

# Check for GPU accelerator
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[OK] GPU Detected  : {len(gpus)} GPU(s) available -> {gpus}")
    try:
        gpu_name = tf.test.gpu_device_name()
        print(f"     Active GPU    : {gpu_name}")
    except Exception:
        pass
    print("     Training will be accelerated using your GPU!")
else:
    print("[WARNING] No GPU detected! TensorFlow is running on CPU.")
    print("          Training 100,000 images on CPU will take multiple hours.")
    print("          TIP: Enable GPU via: Runtime > Change runtime type > T4 GPU > Save.")
print("=" * 60)


## 3. Automated Dataset Acquisition via `kagglehub`
We download the official **CIFAKE** benchmark (120,000 images total, 32x32 pixels, RGB) directly from Kaggle.
- **No manual downloading** or zip file extracting is required.
- **No Kaggle API credentials** (`kaggle.json`) are required when using `kagglehub`.
- The dataset path returned by `kagglehub.dataset_download()` will be automatically resolved and validated.


In [ ]:
# Download CIFAKE dataset automatically via kagglehub
import kagglehub

print("Starting CIFAKE dataset download via kagglehub...")
download_start = time.time()
raw_download_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
download_duration = time.time() - download_start
print(f"[OK] Download completed in {download_duration:.1f} seconds.")
print(f"     Raw path returned by kagglehub: {raw_download_path}")

# Safely locate the directory containing train/FAKE, train/REAL, test/FAKE, test/REAL
def locate_cifake_root(start_path: str | Path) -> Path:
    """
    Scans the downloaded directory (handling direct and nested paths)
    to find the true root containing train and test folders with FAKE and REAL subfolders.
    """
    root = Path(start_path).resolve()
    
    # Check candidates: start_path, subdirectories, and recursive matches
    candidates = [root]
    if root.is_dir():
        candidates.extend([p for p in root.iterdir() if p.is_dir()])
        candidates.extend(list(root.glob("**/CIFAKE")))
    
    for candidate in candidates:
        has_train_fake = (candidate / "train" / "FAKE").is_dir()
        has_train_real = (candidate / "train" / "REAL").is_dir()
        has_test_fake = (candidate / "test" / "FAKE").is_dir()
        has_test_real = (candidate / "test" / "REAL").is_dir()
        
        if has_train_fake and has_train_real and has_test_fake and has_test_real:
            return candidate
            
    raise FileNotFoundError(
        f"Unable to find required CIFAKE directory structure in: {start_path}\n"
        "Expected: train/FAKE, train/REAL, test/FAKE, test/REAL."
    )

DATA_DIR = locate_cifake_root(raw_download_path)
train_dir = DATA_DIR / "train"
test_dir = DATA_DIR / "test"

print("\n" + "=" * 60)
print(f"[OK] Resolved CIFAKE dataset root: {DATA_DIR}")
print("=" * 60)

# Validate image counts per split and class (DO NOT create dummy folders)
def count_images_in_folder(folder_path: Path) -> int:
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    return sum(1 for f in folder_path.iterdir() if f.is_file() and f.suffix.lower() in valid_exts)

train_fake_count = count_images_in_folder(train_dir / "FAKE")
train_real_count = count_images_in_folder(train_dir / "REAL")
test_fake_count = count_images_in_folder(test_dir / "FAKE")
test_real_count = count_images_in_folder(test_dir / "REAL")

print(f"TRAIN split : FAKE = {train_fake_count:,} images | REAL = {train_real_count:,} images | Total = {train_fake_count + train_real_count:,}")
print(f"TEST  split : FAKE = {test_fake_count:,} images | REAL = {test_real_count:,} images | Total = {test_fake_count + test_real_count:,}")
print(f"TOTAL DATASET: {train_fake_count + train_real_count + test_fake_count + test_real_count:,} images")
print("=" * 60)


## 4. Hyperparameter Configuration & Output Directory Setup
We configure the hyperparameters and establish a clean, dedicated output directory (`colab_outputs/`) where all generated models, metrics, and plots will be stored.


In [ ]:
# ----------------------------------------------------------------------
# Hyperparameters & Constants
# ----------------------------------------------------------------------
IMAGE_SIZE = (160, 160)       # Spatial input dimensions expected by MobileNetV2
BATCH_SIZE = 32               # Mini-batch size (reduce to 16 if experiencing GPU memory errors)
SEED = 42                     # Random seed for reproducible dataset splitting and shuffling
LEARNING_RATE = 1e-4          # Adam optimizer learning rate
CLASS_NAMES = ["FAKE", "REAL"] # Canonical class mapping: FAKE = 0, REAL = 1

# Output directory for saving model checkpoints, metrics, and figures
OUTPUT_DIR = Path("colab_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Target Image Size : {IMAGE_SIZE}")
print(f"  Batch Size        : {BATCH_SIZE}")
print(f"  Random Seed       : {SEED}")
print(f"  Learning Rate     : {LEARNING_RATE}")
print(f"  Class Names       : {CLASS_NAMES} -> {dict(enumerate(CLASS_NAMES))}")
print(f"  Output Directory  : {OUTPUT_DIR.resolve()}")


## 5. Dataset Pipeline Construction (Protecting the Test Set)
To ensure rigorous academic evaluation:
1. The **`train/`** directory (100,000 images) is partitioned into:
   - **80% Training Split** (80,000 images) used to train the model weights.
   - **20% Validation Split** (20,000 images) used exclusively for monitoring generalization and early stopping.
2. The **`test/`** directory (20,000 images) is kept **strictly separate and untouched**. It is never used during training or checkpoint selection.


In [ ]:
print("Building tf.data input pipelines...")

# 1. Training dataset (80% of train/ directory)
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,     # Enforces FAKE=0, REAL=1
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

# 2. Validation dataset (20% of train/ directory)
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,     # Enforces FAKE=0, REAL=1
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# 3. Held-out test dataset (completely isolated from training)
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(test_dir),
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,     # Enforces FAKE=0, REAL=1
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# Apply performance optimization: prefetching overlaps data I/O with model execution
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# Inspect a single batch to verify tensor shapes and label values
for sample_images, sample_labels in train_ds.take(1):
    print(f"\n[OK] Sample batch image tensor shape : {sample_images.shape} (Batch, H, W, Channels)")
    print(f"[OK] Sample batch label tensor shape : {sample_labels.shape}")
    print(f"[OK] Pixel value range in batch      : [{sample_images.numpy().min():.1f}, {sample_images.numpy().max():.1f}]")
    print(f"[OK] Sample label values (first 8)   : {sample_labels.numpy()[:8]} (0=FAKE, 1=REAL)")
    break

print("\nDatasets initialized successfully!")


## 6. Model Architecture (MobileNetV2 Transfer Learning)
We implement Transfer Learning using **MobileNetV2** pre-trained on ImageNet:
1. **Input Layer**: Accepts raw RGB pixel values `(160, 160, 3)` in range $[0, 255]$.
2. **Rescaling Layer**: Normalizes $[0, 255]$ pixels to $[-1.0, 1.0]$ via `(x / 127.5) - 1.0` (built into the model so external normalization is not required).
3. **MobileNetV2 Base**: Pre-trained on ImageNet, with feature extraction weights **frozen** (`trainable = False`) to prevent catastrophic forgetting.
4. **Global Average Pooling**: Reduces 2D spatial feature maps into a 1D feature vector.
5. **Dropout Layer (0.2)**: Regularization technique to combat overfitting.
6. **Dense Output (Sigmoid)**: A single output neuron producing $P(\text{REAL}) \in [0.0, 1.0]$.


In [ ]:
def create_mobilenetv2_detector(
    input_shape: Tuple[int, int, int] = (160, 160, 3),
    learning_rate: float = 1e-4,
    dropout_rate: float = 0.2,
) -> tf.keras.Model:
    """
    Constructs and compiles the MobileNetV2 transfer-learning binary classification model.
    """
    inputs = tf.keras.layers.Input(shape=input_shape, name="input_image")
    
    # Internal normalization layer: maps [0, 255] float to [-1, 1]
    x = tf.keras.layers.Rescaling(
        scale=1.0 / 127.5,
        offset=-1.0,
        name="mobilenetv2_rescaling",
    )(inputs)

    # Base feature extractor pretrained on ImageNet
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False  # Freeze pretrained weights

    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="dropout_head")(x)
    
    # Single sigmoid output neuron: P(REAL) in [0, 1]
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="output_sigmoid")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="mobilenetv2_cifake_detector")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model

# Instantiate and summarize model architecture
model = create_mobilenetv2_detector(input_shape=(160, 160, 3), learning_rate=LEARNING_RATE, dropout_rate=0.2)
print("Model Architecture Summary:")
model.summary()


## 7. Model Training with Early Stopping & Checkpointing
We train the model using:
- **`ModelCheckpoint`**: Automatically monitors `val_loss` and saves the best model weights to `colab_outputs/mobilenetv2_cifake_best.keras`.
- **`EarlyStopping`**: Halts training if `val_loss` fails to improve for 3 consecutive epochs (`patience=3`) and restores the best weights.
- **`verbose=1`**: Displays visible epoch progress bars with loss, accuracy, precision, and recall metrics in real time.


In [ ]:
EPOCHS = 5
best_model_path = OUTPUT_DIR / "mobilenetv2_cifake_best.keras"
final_model_path = OUTPUT_DIR / "mobilenetv2_cifake_final.keras"
history_path = OUTPUT_DIR / "training_history.json"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

print("=" * 60)
print(f"STARTING MODEL TRAINING (Max Epochs: {EPOCHS})")
print(f"Best model checkpoint destination: {best_model_path}")
print("=" * 60)

train_start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

train_elapsed = time.time() - train_start_time
print("=" * 60)
print(f"[OK] Training completed in {train_elapsed / 60:.2f} minutes.")
print("=" * 60)

# Save final model state and export training history to JSON
model.save(str(final_model_path))
print(f"Final model saved to: {final_model_path}")

history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(history_dict, f, indent=2)
print(f"Training history saved to: {history_path}")


## 8. Training History Visualization
We plot the training and validation loss curves and accuracy curves across epochs to inspect model convergence and verify that no severe overfitting occurred.


In [ ]:
epochs_range = range(1, len(history.history["loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Loss Curve
ax1.plot(epochs_range, history.history["loss"], "b-o", linewidth=2, label="Training Loss")
ax1.plot(epochs_range, history.history["val_loss"], "r--s", linewidth=2, label="Validation Loss")
ax1.set_title("Training vs. Validation Loss", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Binary Cross-Entropy Loss", fontsize=11)
ax1.legend(loc="upper right")
ax1.grid(True, linestyle="--", alpha=0.6)

# 2. Accuracy Curve
ax2.plot(epochs_range, [acc * 100 for acc in history.history["accuracy"]], "b-o", linewidth=2, label="Training Accuracy (%)")
ax2.plot(epochs_range, [acc * 100 for acc in history.history["val_accuracy"]], "g--^", linewidth=2, label="Validation Accuracy (%)")
ax2.set_title("Training vs. Validation Accuracy", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Accuracy (%)", fontsize=11)
ax2.legend(loc="lower right")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()

# Save figures to output directory
loss_curve_file = FIGURES_DIR / "training_loss_curve.png"
acc_curve_file = FIGURES_DIR / "training_accuracy_curve.png"
fig.savefig(loss_curve_file, dpi=300)
fig.savefig(acc_curve_file, dpi=300)
print(f"[OK] Saved training loss curve to     : {loss_curve_file}")
print(f"[OK] Saved training accuracy curve to : {acc_curve_file}")

plt.show()


## 9. Evaluation on Unseen Held-Out Test Split
We load the best model checkpoint and evaluate it on the 20,000 unseen test images (`test/FAKE` and `test/REAL`).
We compute:
- Overall Test Accuracy & Binary Loss
- Per-Class Precision, Recall, and F1-Score
- Macro-Averaged Performance
- A Full $2 \times 2$ Confusion Matrix
All metrics are exported to JSON and CSV in `colab_outputs/metrics/`.


In [ ]:
import csv

# Load the best checkpoint saved during training
print(f"Loading best model checkpoint from: {best_model_path}...")
eval_model = tf.keras.models.load_model(str(best_model_path))

# 1. Evaluate loss and accuracy
print("\nRunning evaluation on held-out test split (20,000 images)...")
test_eval_results = eval_model.evaluate(test_ds, verbose=1)
metric_names = eval_model.metrics_names
eval_summary = dict(zip(metric_names, [float(v) for v in test_eval_results]))

test_loss = eval_summary.get("loss", 0.0)
test_acc = eval_summary.get("accuracy", 0.0)
print(f"\n[TEST RESULTS] Loss: {test_loss:.4f} | Accuracy: {test_acc * 100:.2f}%")

# 2. Gather predictions across all test batches
print("Generating predictions across test dataset for detailed classification report...")
y_true_list = []
y_pred_probs = []

for batch_images, batch_labels in test_ds:
    batch_preds = eval_model.predict(batch_images, verbose=0).flatten()
    y_pred_probs.extend(batch_preds)
    y_true_list.extend(batch_labels.numpy().flatten())

y_true = np.array(y_true_list, dtype=int)
y_pred_probs = np.array(y_pred_probs, dtype=float)
y_pred = (y_pred_probs >= 0.5).astype(int)

# 3. Compute classification metrics
cls_report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print("\nClassification Report (CIFAKE Test Split):")
print(cls_report)

# Save text classification report
report_txt_file = METRICS_DIR / "classification_report.txt"
with open(report_txt_file, "w", encoding="utf-8") as f:
    f.write(cls_report)

# 4. Compute Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])

# Plot Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix (CIFAKE Test Split)", fontsize=13, fontweight="bold")
plt.colorbar()
tick_marks = np.arange(len(CLASS_NAMES))
plt.xticks(tick_marks, CLASS_NAMES, fontsize=11)
plt.yticks(tick_marks, CLASS_NAMES, fontsize=11)
plt.xlabel("Predicted Class", fontsize=11)
plt.ylabel("Actual True Class", fontsize=11)

thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        val = cm[i, j]
        plt.text(j, i, f"{val:,}", ha="center", va="center",
                 color="white" if val > thresh else "black",
                 fontweight="bold", fontsize=12)

plt.tight_layout()
cm_plot_file = FIGURES_DIR / "confusion_matrix.png"
plt.savefig(cm_plot_file, dpi=300)
print(f"[OK] Saved confusion matrix heatmap to: {cm_plot_file}")
plt.show()

# 5. Export Metrics to JSON and CSV
p_per, r_per, f1_per, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")

metrics_payload = {
    "status": "COMPLETED",
    "dataset": "CIFAKE",
    "evaluation_split": "test",
    "total_test_samples": len(y_true),
    "loss": float(test_loss),
    "accuracy": float(test_acc),
    "macro_avg": {
        "precision": float(p_macro),
        "recall": float(r_macro),
        "f1_score": float(f1_macro),
    },
    "per_class": {
        "FAKE (0)": {"precision": float(p_per[0]), "recall": float(r_per[0]), "f1_score": float(f1_per[0]), "support": int(support[0])},
        "REAL (1)": {"precision": float(p_per[1]), "recall": float(r_per[1]), "f1_score": float(f1_per[1]), "support": int(support[1])},
    },
    "confusion_matrix": {
        "order": ["FAKE (0)", "REAL (1)"],
        "matrix": cm.tolist(),
        "true_negatives_fake": tn,
        "false_positives_real": fp,
        "false_negatives_fake": fn,
        "true_positives_real": tp,
    },
}

metrics_json_file = METRICS_DIR / "metrics.json"
with open(metrics_json_file, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

metrics_csv_file = METRICS_DIR / "metrics.csv"
with open(metrics_csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Metric Category", "Class / Aggregate", "Precision", "Recall", "F1-Score", "Support"])
    writer.writerow(["Per-Class", "FAKE (0)", f"{p_per[0]:.4f}", f"{r_per[0]:.4f}", f"{f1_per[0]:.4f}", int(support[0])])
    writer.writerow(["Per-Class", "REAL (1)", f"{p_per[1]:.4f}", f"{r_per[1]:.4f}", f"{f1_per[1]:.4f}", int(support[1])])
    writer.writerow(["Aggregate", "Macro Average", f"{p_macro:.4f}", f"{r_macro:.4f}", f"{f1_macro:.4f}", len(y_true)])
    writer.writerow(["Overall", "Accuracy", f"{test_acc:.4f}", "-", "-", len(y_true)])

print(f"[OK] Exported metrics JSON to: {metrics_json_file}")
print(f"[OK] Exported metrics CSV  to: {metrics_csv_file}")


## 10. Single-Image Prediction & Demonstration
We test individual images using the trained model to demonstrate single-image inference and explain how confidence scores are calculated.


In [ ]:
def predict_single_image(image_path: str | Path, detector_model: tf.keras.Model, target_size=(160, 160)):
    """
    Loads, preprocesses, and classifies an individual image.
    Outputs the predicted class (REAL or FAKE), confidence percentage, and displays the image.
    """
    path = Path(image_path)
    if not path.is_file():
        print(f"[!] File not found: {path}")
        return
        
    img = Image.open(path).convert("RGB")
    img_resized = img.resize(target_size, Image.Resampling.BILINEAR)
    img_array = np.expand_dims(np.array(img_resized, dtype=np.float32), axis=0) # Shape: (1, 160, 160, 3)

    raw_score = float(detector_model.predict(img_array, verbose=0)[0][0]) # P(REAL)
    
    if raw_score >= 0.5:
        pred_class = "REAL"
        confidence = raw_score * 100.0
    else:
        pred_class = "FAKE"
        confidence = (1.0 - raw_score) * 100.0

    plt.figure(figsize=(3.5, 3.5))
    plt.imshow(img)
    plt.axis("off")
    title_color = "green" if pred_class == "REAL" else "red"
    plt.title(f"Prediction: {pred_class}\nConfidence: {confidence:.2f}%", fontsize=11, fontweight="bold", color=title_color)
    plt.show()

    print(f"Image File       : {path.name}")
    print(f"Predicted Class  : {pred_class}")
    print(f"Model Confidence : {confidence:.2f}%")
    print(f"Raw Sigmoid Score: {raw_score:.4f} (Probability of being REAL)")
    print("-" * 50)

# Demonstrate prediction on 1 sample REAL and 1 sample FAKE image from test directory
sample_real_files = list((test_dir / "REAL").glob("*.jpg"))
sample_fake_files = list((test_dir / "FAKE").glob("*.jpg"))

if sample_real_files and sample_fake_files and 'eval_model' in locals():
    print("DEMO: Testing on an authentic photographic sample (Ground Truth: REAL):")
    predict_single_image(sample_real_files[0], eval_model)
    
    print("\nDEMO: Testing on an AI-generated synthetic sample (Ground Truth: FAKE):")
    predict_single_image(sample_fake_files[0], eval_model)
else:
    print("Sample test images or evaluated model not found.")
